In [ ]:
import nltk
nltk.download('brown')
nltk.download('punkt')
nltk.download('universal_tagset')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


True

In [ ]:
from nltk.corpus import brown

sentences = brown.sents()

# ---------------- PREPROCESS ----------------
print("🔹 Preprocessing...")
data = [[word.lower() for word in sent] for sent in sentences]

# ---------------- N-GRAM ----------------
print("🔹 Training N-gram model...")
from collections import defaultdict

n = 3
ngram_counts = defaultdict(lambda: defaultdict(int))

for sent in data:
    sent = ['<s>'] * (n-1) + sent + ['</s>']
    for i in range(len(sent)-n+1):
        context = tuple(sent[i:i+n-1])
        word = sent[i+n-1]
        ngram_counts[context][word] += 1

ngram_probs = {}
for context in ngram_counts:
    total = sum(ngram_counts[context].values())
    ngram_probs[context] = {
        word: count/total
        for word, count in ngram_counts[context].items()
    }


🔹 Preprocessing...
🔹 Training N-gram model...


In [ ]:
def predict_ngram(context):
    context = tuple(context[-2:])
    if context in ngram_probs:
        return max(ngram_probs[context], key=ngram_probs[context].get)
    return None

# ---------------- HMM ----------------
print("🔹 Training HMM model...")
tagged_sentences = brown.tagged_sents(tagset='universal')

from nltk.tag import hmm
trainer = hmm.HiddenMarkovModelTrainer()
hmm_model = trainer.train_supervised(tagged_sentences[:5000])


🔹 Training HMM model...


In [ ]:
# ---------------- HMM ----------------
print("🔹 Training HMM model...")
tagged_sentences = brown.tagged_sents(tagset='universal')

from nltk.tag import hmm
trainer = hmm.HiddenMarkovModelTrainer()
hmm_model = trainer.train_supervised(tagged_sentences[:5000])

def predict_hmm(sentence):
    tags = hmm_model.tag(sentence)
    last_tag = tags[-1][1]

    word_freq = {}
    for sent in tagged_sentences:
        for word, tag in sent:
            if tag == last_tag:
                word_freq[word] = word_freq.get(word, 0) + 1

    return max(word_freq, key=word_freq.get)

🔹 Training HMM model...


In [ ]:
# ---------------- COMBINED ----------------
def predict(sentence):
    sentence = [w.lower() for w in sentence]

    pred = predict_ngram(sentence)
    if pred:
        return pred
    return predict_hmm(sentence)

In [ ]:
# ---------------- USER INPUT ----------------
print("\n✅ Model Ready!")
while True:
    text = input("\nEnter a sentence (or 'exit'): ")

    if text.lower() == 'exit':
        break

    words = text.split()
    prediction = predict(words)

    print("👉 Next word prediction:", prediction)


✅ Model Ready!

Enter a sentence (or 'exit'): nlp is
👉 Next word prediction: the

Enter a sentence (or 'exit'): nlp is the
👉 Next word prediction: only

Enter a sentence (or 'exit'): nlp is the only
👉 Next word prediction: way

Enter a sentence (or 'exit'): exit
